<a href="https://colab.research.google.com/github/aditya01ad/colab_codes/blob/main/notebooks/OR_Prep/OR_prep_M2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Module 2, Topic 1: Integer Programming Foundations**

**Objective:** Formulate problems with discrete decisions using binary and integer variables. Understand when and why integer programming is harder than LP. Build fluency with PuLP's cat='Binary' and cat='Integer'.

### Why integer programming?

LP assumes variables can be fractional. But in reality:

- You can't build 2.5 warehouses

- You can't hire 3.7 employees

- You can't use a route half the time

> Integer Programming (IP) restricts some or all variables to be whole numbers. If all variables are integer -- pure IP. If some are continuous -> Mixed Integer Programming (MIP). Most real OR problems are MIPs.

In [43]:
!pip install "pulp==2.9.0"

In [44]:
from pulp import *
import numpy as np
import pandas as pd

In [45]:
# LP relaxation vs IP: a quick illustration
# Max 3x + 2y sub to x + y <= 4.5, x,y >= 0

# LP version
print("LP:\n")
lp = LpProblem("LP_version", LpMaximize)
x = LpVariable("x", lowBound=0)
y = LpVariable("y", lowBound=0)

lp += 3*x + 2*y
lp +=   x +   y <= 4.5
lp.solve()
print("LP solution: x =", x.varValue, "y =", y.varValue, "obj =", value(lp.objective))

# IP version
print("\nIP:\n")
ip = LpProblem("IP_version", LpMaximize)
xi = LpVariable("x", lowBound=0, cat='Integer')
yi = LpVariable("y", lowBound=0, cat='Integer')

ip += 3*xi + 2*yi
ip +=   xi + 4*yi  <= 4.5
ip.solve()
print("IP solution: x =", xi.varValue, "y =", yi.varValue, "obj =", value(ip.objective))

LP:

LP solution: x = 4.5 y = 0.0 obj = 13.5

IP:

IP solution: x = 4.0 y = 0.0 obj = 12.0


## The knapsack problem — the "hello world" of IP

> Problem: You have a bag with capacity 15 kg. You can choose from 6 items, each with a weight and value. Maximize total value without exceeding capacity.


| Item |	Weight |	Value |
| ---- | ------- | ------ |
| 1	| 2	| 10 |
| 2	| 3	| 15 |
| 3 |	4	| 20 |
| 4	| 5	| 25 |
| 5	| 7	| 35 |
| 6 |	8 |	40 |

### Formulation:

> * Binary variable x_i = 1 if item i is selected
>* Maximize sum(value_i * x_i)
>* Subject to sum(weight_i * x_i) ≤ capacity
>* x_i ∈ {0,1}

In [46]:
# Knapsack
items = ['I1', 'I2', 'I3', 'I4', 'I5', 'I6']
weights = {'I1': 2, 'I2': 3, 'I3': 4, 'I4': 5, 'I5': 6, 'I6': 7}
values = {'I1': 10, 'I2': 15, 'I3': 20, 'I4': 25, 'I5': 35, 'I6': 40}
capacity = 15

knap = LpProblem("Knapsack", LpMaximize)

# Binary decision variables
x = LpVariable.dicts("select", items, cat='Binary')

# Objective
knap+= lpSum(values[i] * x[i] for i in items)

# Capacity constraints
knap += lpSum(weights[i] * x[i] for i in items) <= capacity

knap.solve()

print("Status:", LpStatus[knap.status])
print("Select items:")
total_w, total_v = 0, 0
for i in items:
  if x[i].varValue > 0.5:
    print(f"  {i}: weight={weights[i]}, value={values[i]}")
    total_w += weights[i]
    total_v += values[i]
print(f"Total weight: {total_w}, Total value: {total_v}")

print("knap:", knap)

Status: Optimal
Select items:
  I1: weight=2, value=10
  I5: weight=6, value=35
  I6: weight=7, value=40
Total weight: 15, Total value: 85
knap: Knapsack:
MAXIMIZE
10*select_I1 + 15*select_I2 + 20*select_I3 + 25*select_I4 + 35*select_I5 + 40*select_I6 + 0
SUBJECT TO
_C1: 2 select_I1 + 3 select_I2 + 4 select_I3 + 5 select_I4 + 6 select_I5
 + 7 select_I6 <= 15

VARIABLES
0 <= select_I1 <= 1 Integer
0 <= select_I2 <= 1 Integer
0 <= select_I3 <= 1 Integer
0 <= select_I4 <= 1 Integer
0 <= select_I5 <= 1 Integer
0 <= select_I6 <= 1 Integer



### Step 3: Logical constraints with binary variables –

**Let's go through the standard patterns:**

> Pattern 1: If-then (x=1 → y=1)
If we open a facility (x=1), we must also hire staff (y=1). Model: x ≤ y.

> Pattern 2: Either-or (at most one of two)
We can either build a warehouse in location A or B, but not both. Model: x_A + x_B ≤ 1.

> Pattern 3: At least one
At least one of three suppliers must be selected. Model: x_A + x_B + x_C ≥ 1.

> Pattern 4: Exactly k
Exactly 2 of 5 projects must be chosen. Model: sum(x_i) = 2

**Problem:** A company is selecting 3 projects from 5. Each project has a cost and profit. The budget is ₹50 lakh. At least 2 projects must be selected. Project 1 and Project 3 are mutually exclusive (can't do both). Project 5 requires Project 2 to be selected first.

In [47]:
projects = ['P1', 'P2', 'P3', 'P4', 'P5']
costs = {'P1': 15, 'P2': 20, 'P3':25, 'P4': 10, 'P5': 30}
profits = {'P1': 30, 'P2': 20, 'P3': 45, 'P4': 20, 'P5': 50}
budget = 50

proj = LpProblem("Project_selection", LpMaximize)
x = LpVariable.dicts("select", projects, cat='Binary')

# Objective: maximize profit
proj += lpSum(profits[i] * x[i] for i in projects)

# Budget constraint
proj += lpSum(costs[i] * x[i] for i in projects) <= budget

# At least 2 project
proj += lpSum(x[i] for i in projects) >= 2

# P1 and P3 mutually exclusive
proj += x['P1'] + x['P3'] <= 1

# P5 reuires P2
proj += x['P5']<= x['P2']

proj.solve()

print("Status:", LpStatus[proj.status])
print("Setected projects:")
for i in projects:
  if x[i].varValue > 0.5:
    print(f"  {i}: cost={costs[i]}, profit={profits[i]}")
print("Total Profit =", value(proj.objective))

Status: Optimal
Setected projects:
  P1: cost=15, profit=30
  P2: cost=20, profit=20
  P4: cost=10, profit=20
Total Profit = 70.0


### Step 4: The facility location problem — a classic MIP

**Problem:** We have 3 potential warehouse locations and 5 customers. Each warehouse has a fixed opening cost and a capacity. Each customer has a demand. Shipping cost from warehouse i to customer j is given. Decide which warehouses to open and how to assign customers to minimize total cost (fixed + shipping).

**Formulation:**

>1. Binary y_i = 1 if warehouse i opened
>2. Continuous x_ij = amount shipped from i to j
>3. Minimize sum(fixed_i * y_i) + sum(cost_ij * x_ij)
>4. Subject to:
>    * sum_i x_ij ≥ demand_j for each customer
>    * sum_j x_ij ≤ capacity_i * y_i for each warehouse
>    * x_ij ≥ 0, y_i ∈ {0,1}

In [48]:
np.random.seed(42)
warehouses = ['W1', 'W2', 'W3']
customers = ['C1', 'C2','C3', 'C4', 'C5']

fixed_cost = {'W1': 100, 'W2': 120, 'W3': 90}
capacity = {'W1':150, 'W2': 200, 'W3': 100}
demand = {'C1': 40, 'C2':60, 'C3': 50, 'C4': 70, 'C5': 30}

# Shipping cost matrix
ship_cost = {
        ('W1', 'C1'): 5, ('W1', 'C2'): 7, ('W1', 'C3'): 6, ('W1', 'C4'): 8, ('W1', 'C5'): 4,
        ('W2', 'C1'): 6, ('W2', 'C2'): 5, ('W2', 'C3'): 8, ('W2', 'C4'): 6, ('W2', 'C5'): 7,
        ('W3', 'C1'): 8, ('W3', 'C2'): 6, ('W3', 'C3'): 5, ('W3', 'C4'): 7, ('W3', 'C5'): 6
        }

fac = LpProblem("Facility_Location", LpMinimize)

# Variables
y = LpVariable.dicts("open", warehouses, cat='Binary')
x = LpVariable.dicts("ship", (warehouses, customers), lowBound=0)

# Objective
fac += lpSum(fixed_cost[i] * y[i] for i in warehouses) + \
       lpSum(ship_cost[(i,j)] * x[i][j] for i in warehouses for j in customers)

# Demand satisfaction
for j in customers:
  fac += lpSum(x[i][j] for i in warehouses) >= demand[j], f"Demand_{j}"

# Capacity (only if open)
for i in warehouses:
  fac += lpSum(x[i][j] for j in customers) <= capacity[i] * y[i], f"Capacity_{i}"

fac.solve()

1

In [49]:
print("Status:", LpStatus[fac.status])
print("Open warehouses:")
for i in warehouses:
  if y[i].varValue > 0.5:
    print(f"  {i} (capacity={capacity[i]}, fixed={fixed_cost[i]})")

print("\nShipping plan:")
for i in warehouses:
  for j in customers:
    if x[i][j].varValue > 0.01:
      print(f"  {i} -> {j}: {x[i][j].varValue:.2f} units at cost {ship_cost[(i,j)]}")

print(f"\nTotal cost = Rs.{value(fac.objective):.2f}")

Status: Optimal
Open warehouses:
  W1 (capacity=150, fixed=100)
  W2 (capacity=200, fixed=120)

Shipping plan:
  W1 -> C1: 40.00 units at cost 5
  W1 -> C3: 50.00 units at cost 6
  W1 -> C5: 30.00 units at cost 4
  W2 -> C2: 60.00 units at cost 5
  W2 -> C4: 70.00 units at cost 6

Total cost = Rs.1560.00


> This is a real OR Analyst problem. We'll encounter variations of this in interviews and on the job.



### Step 5: Big-M and linking constraints — deeper look

> The constraint sum_j x_ij <= capacity_i * y_i is a big-M constraint in disguise. When y_i = 0, the RHS is 0, forcing x_ij = 0. When y_i = 1, the RHS is capacity_i.

> **The general form:** x <= M * y, where M is a large enough upper bound on x. Choosing M too large weakens the LP relaxation (slower solve); too small cuts off feasible solutions. Here, capacity_i is the natural tight upper bound.

_The optimal values may be the same, but the loose M takes longer to solve and is a bad habit. Always use the tightest valid M._

In [53]:
# Same facility location but with M = 10000 instead of capacity
fac_loose = LpProblem("Facility_Loose_M", LpMinimize)
y2 = LpVariable.dicts("open", warehouses, cat='Binary')
x2 = LpVariable.dicts("ship", (warehouses, customers), lowBound=0)

fac_loose += lpSum(fixed_cost[i] * y2[i] for i in warehouses) + \
             lpSum(ship_cost[(i,j)] * x2[i][j] for i in warehouses for j in customers)

for j in customers:
  fac_loose += lpSum(x2[i][j] for i in warehouses) >= demand[j]

M= 10000
for i in warehouses:
  fac_loose += lpSum(x2[i][j] for j in customers) <= M * y2[i]

fac_loose.solve()




1

In [54]:
print("Loose M total ocst:", value(fac_loose.objective))
print("Tight M total cost:", value(fac.objective))

Loose M total ocst: 1560.0
Tight M total cost: 1560.0
